In [1]:
import sys
# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [2]:
TRIALS = 2
FIX_FILE_PATH = "./import_fix.py"
for _ in range(TRIALS):
    try:
        from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
        from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
        from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig, DEFAULT_INMEMORYGRAPH_CONFIG, DEFAULT_NEO4J_CONFIG
        from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
        from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig, DEFAULT_INMEMORYKV_CONFIG, DEFAULT_AEROSPIKE_CONFIG

        from src.qa_pipeline import QAPipelineConfig
        from src.qa_pipeline.query_parser import QueryLLMParserConfig
        from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

        from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
        from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig, AStarMetricsConfig
        from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig
        from src.qa_pipeline.knowledge_retriever.MixturedTripletsRetriever import MixturedGraphSearchConfig

        from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

        from src.memorize_pipeline import MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig

        from src.utils import Logger, NodeType
    except RuntimeError as e:
        from pathlib import Path
        fix_path = Path(FIX_FILE_PATH)
        if fix_path.is_file():
            %run {fix_path} --base_dir BASEDIR
        else:
            raise e

/home/dzigen/Desktop/PersonalAI/pai_venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


#### 1. Задаём конфигурацию графа знаний

In [3]:
# !!! SELECT ONE OF THE STORAGE TYPES !!!

# in-memory storage
GRAPH_STORAGE_CONFIG = GraphDriverConfig(db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)
KV_STORAGE_CONFIG = KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)

# remote storage
#GRAPH_STORAGE_CONFIG = GraphDriverConfig(db_vendor='neo4j', db_config=GraphDBConnectionConfig(
#    uri="bolt://localhost:7687", params={'user': "neo4j", 'pwd': 'password', 'db_name': 'testing'}))
#KV_STORAGE_CONFIG = KeyValueDriverConfig(db_vendor='aerospike', db_config=DEFAULT_AEROSPIKE_CONFIG)

# !!! SELECT ONE OF THE STORAGE TYPES !!!

In [4]:
#
LANGUAGE = 'auto' # 'ru' , 'en', 'auto

#
RETRIEVER_NAME = 'mixture'
RETRIEVER_HYPERP = MixturedGraphSearchConfig(
    astar_config=AStarGraphSearchConfig(
        metrics_config=AStarMetricsConfig(
        h_metric_name='ip'), # 'ip' , 'weight_with_short_path', 'avg_weighted_with_short_path'
        max_depth=20, max_passed_nodes=1000,
        accepted_node_types=[NodeType.object , NodeType.hyper, NodeType.episodic]),
    bfs_config=BFSSearchConfig(
        strict_filter = True, hyper_episodic_num = 15,
        chain_triplets_num = 25, other_triplets_num = 6)
)

# embedder hyperp
DEVICE = 'cuda'
EMBEDDER_MODEL_PATH = '../../models/intfloat/multilingual-e5-small'

# vector dbs hyperp
NODES_DB_PATH = '../../data/graph_structures/vectorized_nodes/testing'
TRIPLETS_DB_PATH = '../../data/graph_structures/vectorized_triplets/testing'
NEED_TO_CLEAR = True

In [5]:
inmemory_kg_config = RemoteKnowledgeGraphConfig(
    graph_struct_config=GraphModelConfig(driver_config=GRAPH_STORAGE_CONFIG),
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=NODES_DB_PATH, db_name='vectorized_nodes', need_to_clear=NEED_TO_CLEAR)),
        tripletsdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=TRIPLETS_DB_PATH, db_name='vectorized_triplets', need_to_clear=NEED_TO_CLEAR)),
        embedder_config=EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH, device=DEVICE)),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(lang=LANGUAGE),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method=RETRIEVER_NAME,retriever_config=RETRIEVER_HYPERP,
            cache_config=KV_STORAGE_CONFIG),
        answer_generator_config=QALLMGeneratorConfig(lang=LANGUAGE)),
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(lang=LANGUAGE),
        updator_config=LLMUpdatorConfig(lang=LANGUAGE)),
    log=Logger('log/main'))

#### 2. Инициализируем граф знаний

In [6]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


OutOfMemoryError: CUDA out of memory. Tried to allocate 368.00 MiB. GPU 0 has a total capacty of 3.94 GiB of which 64.00 MiB is free. Process 17739 has 2.49 GiB memory in use. Process 657716 has 546.00 MiB memory in use. Process 661286 has 546.00 MiB memory in use. Including non-PyTorch memory, this process has 44.00 MiB memory in use. Of the allocated memory 0 bytes is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [7]:
# !!! CLEANING REMOTE GRAPH !!!
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) -[r] -> () delete a, r")
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) delete a")
# !!! CLEANING REMOTE GRAPH !!!

#### 3. Добавляем в граф информацию

In [ ]:
messages = [
    "Mikhail Menshchikov is currently a second-year master's student at ITMO.",
    "Mikhail Menshchikov is studying in the Master's program 'Deep Learning and Generative AI'",
    "Mikhail Menshchikov completed his bachelor's degree at Petrozavodsk State University",
    "Petrozavodsk State University is where Mikhail Menshchikov received his bachelor's degree.",
    "Mikhail Menshchikov studied at Petrozavodsk State University and received a bachelor's degree."]
properties = [dict() for _ in range(len(messages))]

triplets, info = [], []
for text, prop in zip(messages, properties):
    tmp_triplets, tmp_info = rkg_main.update_memory(text, prop)
    triplets.append(tmp_triplets)
    info.append(tmp_info)

#### 4. Q&A

In [9]:
answer, info = rkg_main.answer_question("What program is Mikhail Menshchikov studying for his master's degree?")
print(answer)

Deep Learning and Generative AI


In [10]:
answer, info = rkg_main.answer_question("Where did Mikhail Menshchikov receive his bachelor's degree?")
print(answer)

Petrozavodsk State University
